# Notebook 01: Data Loading and Cleaning

**Project F1: Electricity Consumption Analysis and Forecasting in Morocco**

## Objective

Load the Tetouan electricity-consumption dataset, check its quality, and save a consistent dataset for later analysis.

This notebook covers:

- Loading the raw CSV through our shared data loader.
- Checking column names, data types, and missing values.
- Verifying timestamps, duplicates, and ten-minute spacing.
- Profiling electricity demand and weather measurements.
- Saving the checked dataset and documenting its structure.

## Project context

Our research question is:

**How does electricity demand vary with time and weather, and can peak demand be anticipated?**

The dataset contains electricity-consumption measurements for three zones, alongside weather measurements.

**Source:** [UCI Machine Learning Repository — Power Consumption of Tetouan City](https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city)

We preserve the original CSV. Any standardisation happens in memory, and the resulting dataset is saved separately.

### Variable Dictionary
Below is the official data dictionary from the UCI repository. As confirmed by our upcoming checks, there are no missing values in any column.

![UCI Tetouan Dataset Variables](dataset_variables.png)


## 1. Environment and Project Paths

Our shared Python files live in `src/`:

- `config.py` defines paths and shared settings.
- `data_loader.py` reads the CSV, standardises column names, parses timestamps, and sorts the observations.

The setup below finds the project folder by looking for these files. This supports running the notebook from either the project root or the `notebooks/` folder.

We reload the shared modules so that recent edits are available in the current notebook session.

In [11]:
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Find the project root from the current folder or one of its parents.
current_dir = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        folder
        for folder in [current_dir, *current_dir.parents]
        if (folder / "src" / "config.py").is_file()
        and (folder / "src" / "data_loader.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project folder containing "
        "src/config.py and src/data_loader.py. "
        "Open the Project folder in VS Code and try again."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as cfg
importlib.reload(cfg)

import src.data_loader as dl
importlib.reload(dl)

from src.config import DATA_RAW, PROCESSED_DIR, TABLES_DIR, ZONES
from src.data_loader import load_raw

print("Python environment:", sys.executable)
print("Project root      :", PROJECT_ROOT)
print("Configuration     :", cfg.__file__)
print("Raw CSV           :", DATA_RAW)
print("Raw file exists   :", DATA_RAW.is_file())

assert DATA_RAW.is_file(), f"Raw CSV not found: {DATA_RAW}"
assert list(ZONES) == ["zone1", "zone2", "zone3"], (
    f"Unexpected zone configuration: {ZONES}"
)

Python environment: c:\Users\banda\Desktop\MScBA\Semester_1\Descriptive Analytics & Data Visualization\Project\.venv\Scripts\python.exe
Project root      : C:\Users\banda\Desktop\MScBA\Semester_1\Descriptive Analytics & Data Visualization\Project
Configuration     : C:\Users\banda\Desktop\MScBA\Semester_1\Descriptive Analytics & Data Visualization\Project\src\config.py
Raw CSV           : C:\Users\banda\Desktop\MScBA\Semester_1\Descriptive Analytics & Data Visualization\Project\Dataset\Tetuan_City_power_consumption.csv
Raw file exists   : True


### Interpretation: Environment and Paths

The setup successfully located the project folder and the raw CSV.

The Python environment should point to this project's `.venv`. This confirms that the notebook uses the environment where our project dependencies are installed.

The configuration path should point to our own `src/config.py`, and `Raw file exists` should be `True`.

These checks establish that we are loading the intended project files before analysing the data.

## 2. Load and Inspect the Dataset

The data loader performs four standardisation steps:

1. Removes extra whitespace from column names.
2. Renames columns to consistent lowercase names.
3. Converts the timestamp column into a datetime type.
4. Sorts observations chronologically.

It does not fill missing values, remove observations, or change measurement values.

We inspect the first and last observations to understand the structure and time coverage.



In [12]:
df = load_raw()

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"First timestamp: {df['datetime'].min()}")
print(f"Last timestamp : {df['datetime'].max()}")

display(df.head())
display(df.tail())

df.info()

Rows: 52,416
Columns: 9
First timestamp: 2017-01-01 00:00:00
Last timestamp : 2017-12-30 23:50:00


,datetime,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1,zone2,zone3
0,2017-01-01 00:00:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386
1,2017-01-01 00:10:00,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434
2,2017-01-01 00:20:00,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373
3,2017-01-01 00:30:00,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711
4,2017-01-01 00:40:00,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964


,datetime,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1,zone2,zone3
52411,2017-12-30 23:10:00,7.010,72.4,0.080,0.040,0.096,31160.45627,26857.31820,14780.31212
52412,2017-12-30 23:20:00,6.947,72.6,0.082,0.051,0.093,30430.41825,26124.57809,14428.81152
52413,2017-12-30 23:30:00,6.900,72.8,0.086,0.084,0.074,29590.87452,25277.69254,13806.48259
52414,2017-12-30 23:40:00,6.758,73.0,0.080,0.066,0.089,28958.17490,24692.23688,13512.60504
52415,2017-12-30 23:50:00,6.580,74.1,0.081,0.062,0.111,28349.80989,24055.23167,13345.49820


<class 'pandas.DataFrame'>
RangeIndex: 52416 entries, 0 to 52415
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   datetime               52416 non-null  datetime64[us]
 1   temperature            52416 non-null  float64       
 2   humidity               52416 non-null  float64       
 3   wind_speed             52416 non-null  float64       
 4   general_diffuse_flows  52416 non-null  float64       
 5   diffuse_flows          52416 non-null  float64       
 6   zone1                  52416 non-null  float64       
 7   zone2                  52416 non-null  float64       
 8   zone3                  52416 non-null  float64       
dtypes: datetime64[us](1), float64(8)
memory usage: 3.6 MB


### Interpretation: Dataset Structure

The dataset contains **52,416 rows and nine columns**:

- One timestamp column.
- Five weather measurement columns.
- Three electricity-demand columns, one for each zone.

The first observation is **January 1, 2017 at 00:00**, and the last is **December 30, 2017 at 23:50**.

The timestamp column has been converted into a datetime type, and all eight measurement columns are numeric. In our current environment, the timestamp type is displayed as `datetime64[us]`, where `us` indicates microsecond precision.

The first and last rows provide an initial view of the data. The following checks establish whether the full dataset has the expected structure and coverage.

## 3. Validate the Schema and Measurement Values

We expect one timestamp column, five weather columns, and three electricity-demand columns.

These checks confirm that:

- All expected columns are present, with no unexpected columns.
- Timestamps have a datetime type.
- Measurement columns are numeric.
- No columns contain missing values.
- Numeric values are finite, meaning they contain neither positive nor negative infinity.

Checks use assertions so that unexpected data stops the workflow before we save a processed file.

In [13]:
EXPECTED_COLUMNS = [
    "datetime",
    "temperature",
    "humidity",
    "wind_speed",
    "general_diffuse_flows",
    "diffuse_flows",
    "zone1",
    "zone2",
    "zone3",
]

NUMERIC_COLUMNS = EXPECTED_COLUMNS[1:]

assert df.columns.is_unique, "Duplicate column names found."

assert set(df.columns) == set(EXPECTED_COLUMNS), (
    f"Missing columns: {set(EXPECTED_COLUMNS) - set(df.columns)}; "
    f"Unexpected columns: {set(df.columns) - set(EXPECTED_COLUMNS)}"
)

# Apply a consistent column order for downstream notebooks.
df = df[EXPECTED_COLUMNS].copy()

assert pd.api.types.is_datetime64_any_dtype(df["datetime"]), (
    "The datetime column has not been parsed correctly."
)

for column in NUMERIC_COLUMNS:
    assert pd.api.types.is_numeric_dtype(df[column]), (
        f"{column} must be numeric."
    )

missing_summary = df.isna().sum().rename("missing_values").to_frame()
display(missing_summary)

assert not df.isna().any().any(), "Missing values found."

assert np.isfinite(df[NUMERIC_COLUMNS].to_numpy()).all(), (
    "Infinite measurement values found."
)

print("Schema, missing-value, and finite-value checks passed.")

,missing_values
datetime,0
temperature,0
humidity,0
wind_speed,0
general_diffuse_flows,0
diffuse_flows,0
zone1,0
zone2,0
zone3,0


Schema, missing-value, and finite-value checks passed.


### Interpretation: Schema and Completeness

All nine expected columns are present, with no unexpected or duplicate column names.

The missing-value table reports **zero missing values in every column**. All measurement columns are numeric and contain finite values.

Therefore, no missing-value imputation is needed for this file.

These checks confirm structural completeness. They do not establish that every recorded measurement is accurate.

## 4. Verify Time Coverage and Observation Frequency

The current CSV is expected to contain **52,416 observations**, covering **January 1 to December 30, 2017**.

At ten-minute intervals, a complete day contains:

**24 hours × 6 observations per hour = 144 observations**

We check the exact start and end timestamps, timestamp uniqueness, chronological ordering, and spacing between consecutive observations.

We also compare the observed dates with the full 2017 calendar. This distinguishes gaps inside the observed period from calendar dates outside the file's coverage.

In [14]:
EXPECTED_START = pd.Timestamp("2017-01-01 00:00:00")
EXPECTED_END = pd.Timestamp("2017-12-30 23:50:00")
EXPECTED_ROWS = 52_416
EXPECTED_INTERVAL = pd.Timedelta(minutes=10)

assert df.shape == (EXPECTED_ROWS, len(EXPECTED_COLUMNS)), (
    f"Unexpected dataset shape: {df.shape}"
)

assert df["datetime"].min() == EXPECTED_START, (
    f"Unexpected start timestamp: {df['datetime'].min()}"
)

assert df["datetime"].max() == EXPECTED_END, (
    f"Unexpected end timestamp: {df['datetime'].max()}"
)

assert df["datetime"].is_monotonic_increasing, (
    "Timestamps are not sorted chronologically."
)

duplicate_timestamps = int(df["datetime"].duplicated().sum())
assert duplicate_timestamps == 0, (
    f"Found {duplicate_timestamps} duplicate timestamps."
)

time_differences = df["datetime"].diff().dropna()
assert time_differences.eq(EXPECTED_INTERVAL).all(), (
    "Unexpected intervals between observations:\n"
    f"{time_differences.value_counts().head()}"
)

expected_timestamps = pd.date_range(
    EXPECTED_START,
    EXPECTED_END,
    freq="10min",
)

observed_timestamps = pd.DatetimeIndex(df["datetime"])
missing_timestamps = expected_timestamps.difference(observed_timestamps)

assert len(missing_timestamps) == 0, (
    f"Missing timestamps within the observed period: "
    f"{missing_timestamps[:10].tolist()}"
)

daily_counts = df.groupby(df["datetime"].dt.normalize()).size()
assert daily_counts.eq(144).all(), "Some observed days are incomplete."

calendar_days = pd.date_range("2017-01-01", "2017-12-31", freq="D")
observed_days = pd.DatetimeIndex(
    df["datetime"].dt.normalize().unique()
)
missing_calendar_days = calendar_days.difference(observed_days)

coverage_summary = pd.DataFrame(
    {
        "check": [
            "Rows",
            "Observed days",
            "Duplicate timestamps",
            "Missing timestamps within observed period",
            "First timestamp",
            "Last timestamp",
            "Calendar dates absent from file",
        ],
        "result": [
            len(df),
            len(observed_days),
            duplicate_timestamps,
            len(missing_timestamps),
            str(df["datetime"].min()),
            str(df["datetime"].max()),
            ", ".join(missing_calendar_days.strftime("%Y-%m-%d")),
        ],
    }
)

display(coverage_summary)

print("Time coverage and ten-minute spacing checks passed.")

,check,result
0,Rows,52416
1,Observed days,364
2,Duplicate timestamps,0
3,Missing timestamps within observed period,0
4,First timestamp,2017-01-01 00:00:00
5,Last timestamp,2017-12-30 23:50:00
6,Calendar dates absent from file,2017-12-31


Time coverage and ten-minute spacing checks passed.


### Interpretation of Time Coverage

If the checks above pass, the file contains 364 complete days with uninterrupted ten-minute observations between January 1 and December 30, 2017.

**December 31 is absent from the file.** We therefore describe the dataset as covering January 1–December 30, rather than a complete calendar year.

We will not create artificial observations for the missing final day. Later analyses and forecasting evaluations will use the available period, and the report will document this limitation.

The timestamps contain no timezone information. We preserve them as recorded and do not assign a timezone or apply daylight-saving adjustments in this notebook.

## 5. Profile Measurements and Flag Potential Anomalies

We inspect summary statistics and screen for:

- Negative zone-demand values.
- Humidity outside the range 0–100%.
- Negative wind-speed values.
- Unusually large deviations in zone demand.
- Long runs of identical consecutive measurements.

These are screening checks. Passing them does not prove that sensors operated correctly or that every observation is accurate.

A Z-score measures distance from a column's mean in standard deviations. We count zone-demand observations with an absolute Z-score above 6 as one simple diagnostic. Seasonal or local anomalies may require other methods.

We do not automatically remove or clip observations.

In [15]:
display(df[NUMERIC_COLUMNS].describe().T.round(3))

range_flags = pd.Series(
    {
        **{
            f"{zone}: negative demand": int((df[zone] < 0).sum())
            for zone in ZONES
        },
        "humidity: outside 0–100%": int(
            (~df["humidity"].between(0, 100)).sum()
        ),
        "wind_speed: negative values": int(
            (df["wind_speed"] < 0).sum()
        ),
    },
    name="flagged_observations",
)

display(range_flags.to_frame())

assert range_flags.eq(0).all(), (
    "Potential range issues found. Investigate before saving."
)

# Simple global outlier screening for zone demand.
zone_std = df[ZONES].std(ddof=0)
assert zone_std.gt(0).all(), "A zone has zero variation."

zone_z_scores = (
    df[ZONES] - df[ZONES].mean()
) / zone_std

extreme_counts = (
    zone_z_scores.abs()
    .gt(6)
    .sum()
    .rename("observations_with_absolute_z_score_above_6")
)

display(extreme_counts.to_frame())


def longest_identical_run(series):
    """Return the largest number of consecutive identical readings."""
    run_groups = series.ne(series.shift()).cumsum()
    return int(series.groupby(run_groups).size().max())


run_summary = pd.DataFrame(
    {
        "column": NUMERIC_COLUMNS,
        "longest_identical_run": [
            longest_identical_run(df[column])
            for column in NUMERIC_COLUMNS
        ],
    }
)

# N consecutive readings span N - 1 ten-minute intervals.
run_summary["elapsed_minutes_first_to_last"] = (
    run_summary["longest_identical_run"] - 1
) * 10

display(run_summary)

,count,mean,std,min,25%,50%,75%,max
temperature,52416.0,18.810,5.815,3.247,14.410,18.780,22.890,40.010
humidity,52416.0,68.260,15.551,11.340,58.310,69.860,81.400,94.800
wind_speed,52416.0,1.959,2.349,0.050,0.078,0.086,4.915,6.483
general_diffuse_flows,52416.0,182.697,264.401,0.004,0.062,5.036,319.600,1163.000
diffuse_flows,52416.0,75.028,124.211,0.011,0.122,4.456,101.000,936.000
zone1,52416.0,32344.971,7130.563,13895.696,26310.669,32265.920,37309.018,52204.395
zone2,52416.0,21042.509,5201.466,8560.081,16980.766,20823.168,24713.718,37408.861
zone3,52416.0,17835.406,6622.165,5935.174,13129.327,16415.117,21624.100,47598.326


,flagged_observations
zone1: negative demand,0
zone2: negative demand,0
zone3: negative demand,0
humidity: outside 0–100%,0
wind_speed: negative values,0


,observations_with_absolute_z_score_above_6
zone1,0
zone2,0
zone3,0


,column,longest_identical_run,elapsed_minutes_first_to_last
0,temperature,4,30
1,humidity,9,80
2,wind_speed,7,60
3,general_diffuse_flows,5,40
4,diffuse_flows,5,40
5,zone1,3,20
6,zone2,3,20
7,zone3,3,20


### Interpretation: Measurement Profiles

Temperature ranges from approximately **3.25°C to 40.01°C**, while humidity ranges from **11.34% to 94.8%**.

The range checks found:

- No negative electricity-demand values in any zone.
- No humidity values outside 0–100%.
- No negative wind-speed values.

The average recorded demand differs across zones:

| Zone | Mean demand, in source units |
|---|---:|
| Zone 1 | 32,344.97 |
| Zone 2 | 21,042.51 |
| Zone 3 | 17,835.41 |

Zone 1 has the highest average demand over the observed period. We will investigate how these differences change by hour, weekday, and month in Notebook 02.

### Interpretation: Extreme-Value Screening

No zone-demand observations have an absolute global Z-score above 6.

This means that the screening method did not flag exceptionally large deviations from each zone's overall mean. It does not rule out local anomalies, seasonal changes, or other unusual patterns.

We retain all observations at this stage.

### Interpretation: Repeated Measurements

The longest runs of identical consecutive measurements are:

| Variable | Consecutive readings | Time from first to last reading |
|---|---:|---:|
| Temperature | 4 | 30 minutes |
| Humidity | 9 | 80 minutes |
| Wind speed | 7 | 60 minutes |
| General diffuse flows | 5 | 40 minutes |
| Diffuse flows | 5 | 40 minutes |
| Zone 1 | 3 | 20 minutes |
| Zone 2 | 3 | 20 minutes |
| Zone 3 | 3 | 20 minutes |

The elapsed time uses **(number of readings − 1) × 10 minutes**. For example, three readings at 00:00, 00:10, and 00:20 span 20 minutes.

Repeated values may reflect stable conditions, measurement precision, or recording behaviour. These runs alone do not prove a sensor fault, so we do not remove them.

## 6. Inspect the Wind-Speed Distribution

Wind speed deserves closer inspection because many observations cluster near low values, while another group occurs at higher values.

We report quantiles and the proportions within two descriptive ranges. These ranges help describe this file; they are not established fault thresholds.

A clustered distribution alone does not establish a sensor problem. We retain the readings and will explore their relationship with demand in Notebook 03.

In [16]:
wind_quantiles = (
    df["wind_speed"]
    .quantile([0, 0.25, 0.50, 0.75, 0.95, 1])
    .rename("wind_speed")
    .to_frame()
)

display(wind_quantiles)

wind_summary = pd.DataFrame(
    {
        "description": [
            "Wind-speed readings below 0.1",
            "Wind-speed readings between 4 and 6, inclusive",
        ],
        "percentage_of_observations": [
            df["wind_speed"].lt(0.1).mean() * 100,
            df["wind_speed"].between(4, 6).mean() * 100,
        ],
    }
)

display(wind_summary.round(2))

print("Wind-speed values are retained without modification.")

,wind_speed
0.00,0.050
0.25,0.078
0.50,0.086
0.75,4.915
0.95,4.923
1.00,6.483


,description,percentage_of_observations
0,Wind-speed readings below 0.1,57.01
1,"Wind-speed readings between 4 and 6, inclusive",38.62


Wind-speed values are retained without modification.


### Interpretation: Wind-Speed Distribution

The wind-speed quantiles show a marked separation between lower and higher readings:

- The median is **0.086**.
- The 75th percentile is **4.915**.
- The maximum is **6.483**.

Approximately **57.01%** of readings are below 0.1, while **38.62%** fall between 4 and 6, inclusive. The remaining **4.37%** lie outside those two ranges.

These results describe a concentration of readings in two distinct ranges. A histogram in the weather-analysis notebook will help us inspect the distribution more clearly.

We cannot conclude from these figures alone that the low readings represent a sensor floor or that either group is faulty. We retain all wind-speed observations and will assess their usefulness for forecasting later.

## 7. Document the Processed Dataset

The data dictionary records the names, types, and meanings of the columns used in later notebooks.

Measurement units should be checked against authoritative source documentation before we label charts or convert power measurements into energy.

For now:

- Temperature is recorded in degrees Celsius.
- Humidity is recorded as a percentage.
- Wind speed, flow measurements, and zone demand retain their source values.
- Units requiring confirmation are explicitly marked.
- Timestamps remain timezone-naive, as supplied.

This dictionary documents the schema. The assertions earlier in the notebook enforce the structural checks.

In [17]:
data_dictionary = pd.DataFrame(
    {
        "column": EXPECTED_COLUMNS,
        "dtype": [
            str(df[column].dtype)
            for column in EXPECTED_COLUMNS
        ],
        "description": [
            "Observation timestamp",
            "Recorded air temperature",
            "Recorded relative humidity",
            "Recorded wind speed",
            "Recorded general diffuse flows",
            "Recorded diffuse flows",
            "Recorded electricity demand in Zone 1",
            "Recorded electricity demand in Zone 2",
            "Recorded electricity demand in Zone 3",
        ],
        "unit_or_time_note": [
            "Timezone-naive; ten-minute intervals",
            "Degrees Celsius",
            "Percent",
            "Source unit requires confirmation",
            "Source unit requires confirmation",
            "Source unit requires confirmation",
            "Source unit requires confirmation",
            "Source unit requires confirmation",
            "Source unit requires confirmation",
        ],
    }
)

display(data_dictionary)

,column,dtype,description,unit_or_time_note
0,datetime,datetime64[us],Observation timestamp,Timezone-naive; ten-minute intervals
1,temperature,float64,Recorded air temperature,Degrees Celsius
2,humidity,float64,Recorded relative humidity,Percent
3,wind_speed,float64,Recorded wind speed,Source unit requires confirmation
4,general_diffuse_flows,float64,Recorded general diffuse flows,Source unit requires confirmation
5,diffuse_flows,float64,Recorded diffuse flows,Source unit requires confirmation
6,zone1,float64,Recorded electricity demand in Zone 1,Source unit requires confirmation
7,zone2,float64,Recorded electricity demand in Zone 2,Source unit requires confirmation
8,zone3,float64,Recorded electricity demand in Zone 3,Source unit requires confirmation


### Interpretation: Data Dictionary

The dictionary documents all nine processed columns, including their actual data types and descriptions.

Temperature and humidity are documented in degrees Celsius and percent. The units for wind speed, flow measurements, and zone demand remain explicitly marked for confirmation against source documentation.

Until those units are confirmed, we will describe zone-demand values as **source units**. We will avoid labelling them as energy consumption or converting them into energy totals.

The timestamps remain timezone-naive. We preserve the recorded times without assigning an unverified timezone.

This documentation helps team members use consistent column names and distinguish verified information from assumptions.

## 8. Save and Verify the Outputs

We save the processed dataset in Parquet format because it preserves column types and supports efficient storage.

We also save the data dictionary and quality summaries so that team members can review the checks without rerunning the notebook.

Finally, we reload the Parquet file and compare it with the in-memory dataframe. This checks that saving and loading preserve the data.

Running this section again replaces the generated outputs with the latest checked versions. The raw CSV remains unchanged.

In [18]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

clean_path = PROCESSED_DIR / "tetuan_clean.parquet"

df.to_parquet(clean_path, index=False, engine="pyarrow")

# Verify that the saved file preserves values and data types.
saved_df = pd.read_parquet(clean_path, engine="pyarrow")
pd.testing.assert_frame_equal(df, saved_df)

data_dictionary.to_csv(
    PROCESSED_DIR / "data_dictionary.csv",
    index=False,
    encoding="utf-8",
)

# Replace the earlier text contract with the current documentation.
(PROCESSED_DIR / "column_contract.txt").write_text(
    data_dictionary.to_string(index=False),
    encoding="utf-8",
)

coverage_summary.to_csv(
    TABLES_DIR / "01_time_coverage.csv",
    index=False,
    encoding="utf-8",
)

range_flags.to_csv(
    TABLES_DIR / "01_range_checks.csv",
    encoding="utf-8",
)

extreme_counts.to_csv(
    TABLES_DIR / "01_extreme_value_counts.csv",
    encoding="utf-8",
)

run_summary.to_csv(
    TABLES_DIR / "01_identical_runs.csv",
    index=False,
    encoding="utf-8",
)

wind_summary.to_csv(
    TABLES_DIR / "01_wind_distribution_summary.csv",
    index=False,
    encoding="utf-8",
)

print("Processed dataset saved and verified.")
print("Location:", clean_path)
print(f"File size: {clean_path.stat().st_size / 1024**2:.2f} MB")
print(f"Rows retained: {len(saved_df):,}")
print("Data dictionary:", PROCESSED_DIR / "data_dictionary.csv")
print("Quality summaries:", TABLES_DIR)

Processed dataset saved and verified.
Location: C:\Users\banda\Desktop\MScBA\Semester_1\Descriptive Analytics & Data Visualization\Project\outputs\processed\tetuan_clean.parquet
File size: 1.69 MB
Rows retained: 52,416
Data dictionary: C:\Users\banda\Desktop\MScBA\Semester_1\Descriptive Analytics & Data Visualization\Project\outputs\processed\data_dictionary.csv
Quality summaries: C:\Users\banda\Desktop\MScBA\Semester_1\Descriptive Analytics & Data Visualization\Project\outputs\tables


### Interpretation: Saved Outputs

The processed dataset was saved successfully as `tetuan_clean.parquet`, with a file size of approximately **1.69 MB**.

The file was then reloaded and compared with the in-memory dataframe. The comparison passed, confirming that the saved file preserves the dataframe's values, column order, and data types.

All **52,416 observations** were retained.

The notebook also saved:

- A CSV data dictionary and an updated text column contract.
- A time-coverage summary.
- Measurement-range checks.
- Extreme-value screening counts.
- Summaries of identical consecutive readings.
- A wind-speed distribution summary.

The processed dataset is ready for Notebook 02. Future notebooks can load this file instead of repeating the raw-data preparation.

## 9. Conclusion and Next Step

Successful execution confirms that:

- The dataset contains 52,416 observations and nine columns.
- The observed period is January 1–December 30, 2017.
- Timestamps are unique and follow a continuous ten-minute sequence.
- Every observed day contains 144 readings.
- All columns are complete and measurement values are finite.
- The processed dataset survives a save-and-reload comparison.

We standardised column names, parsed timestamps, and sorted observations. We did not remove rows, fill missing dates, or alter measurement values.

The anomaly screens provide initial evidence about data quality. They do not establish that all readings are error-free.


### Remaining limitations

- December 31 is absent.
- The data covers one city and one annual period.
- Timestamps contain no timezone metadata.
- Some measurement units still require source verification.
- The wind-speed distribution warrants further exploration.

### Next notebook

**Notebook 02: Exploratory Analysis of Demand Patterns**

We will examine how demand differs across hours, weekdays, months, and zones. We will distinguish average demand profiles from individual maximum observations.
